In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import PIIMiddleware, ModelRequest, ModelResponse, ToolCallRequest
from langchain.agents.middleware import after_agent, after_model, before_agent, before_model, wrap_model_call, wrap_tool_call
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime
from pydantic import SecretStr
from typing import Callable, List


api_key = SecretStr(userdata.get('OPENAI_API_KEY'))


def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
@tool
def get_interesting_fact() -> str:
    """
    This tool will discover an interesting fact to you.
    """

    return "The Earth is actually not a perfect sphere."

# NOTE: Middlewares can be implemented by separate functions or by a single class inheriting from `AgentMiddleware`.
@before_agent
def before_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_agent")
    print(state)
    print(runtime)

@before_model
def before_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_model")
    print(state)
    print(runtime)

@after_model
def after_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_model")
    print(state)
    print(runtime)

@after_agent
def after_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_agent")
    print(state)
    print(runtime)

@wrap_model_call
def handle_model_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
    print("Event: model_call")
    print(request)

    # Only force tool on the first LLM step
    if len(request.messages) == 1:
        print("Forcing a specific tool call")
        request = request.override(tool_choice=get_interesting_fact.name)

    response = handler(request)

    print("Obtained model response:")
    print(response)

    return response

@wrap_tool_call
def handle_tool_call(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage]):
    print("Event: tool_call")
    print(request)

    response = handler(request)

    print("Obtained tool message:")
    print(response)

    return response

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key),
    tools=[get_interesting_fact],
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact"),
        before_agent_func,
        before_model_func,
        after_agent_func,
        after_model_func,
        handle_model_call,
        handle_tool_call
    ]
)

In [ ]:
reserve_ticket = agent.invoke(
    input={
        "messages": [HumanMessage("Book two seats under maria.popova@example.com for the play at the theater this evening.")]
    }
)

In [ ]:
print_conversation(reserve_ticket["messages"])